# Kaggle — Nucleus / final-learner alignment diagnostic

This notebook reuses the existing DINO cache, nucleus train cache, and selected-index checkpoints. It does **not** rerun CellViT or acquisition. Evaluation is on the unselected training pool, so the result is a go/no-go diagnostic rather than official test accuracy.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPO_URL = 'https://github.com/CryAndRRich/codapath.git'
REPO_BRANCH = 'tiendung'
CODAPATH = Path('/kaggle/working/codapath')
if (CODAPATH / '.git').is_dir():
    subprocess.check_call(['git', '-C', str(CODAPATH), 'fetch', 'origin', REPO_BRANCH])
    subprocess.check_call(['git', '-C', str(CODAPATH), 'switch', REPO_BRANCH])
    subprocess.check_call(['git', '-C', str(CODAPATH), 'pull', '--ff-only', 'origin', REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f'{CODAPATH} exists but is not a Git repository')
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(CODAPATH)])
os.chdir(CODAPATH)
actual_branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print('repo:', CODAPATH, '| branch:', actual_branch)

In [ ]:
# Install only packages missing from the Kaggle image. CellViT is not needed.
import importlib.util
needed = {
    'yaml': 'PyYAML>=6.0',
    'sklearn': 'scikit-learn>=1.3.0',
    'torchvision': 'torchvision',
    'transformers': 'transformers>=4.27.0',
}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('dependencies ready')

In [ ]:
# ---- EDIT ONLY THIS CELL WHEN AUTO-DETECTION IS AMBIGUOUS ----
DATASET = 'pathmnist'
SEED = 42
SELECTION_RUN = 'nucleus_cellvit_embedding_disagreement'
ALL_RUNS = False

# Leave these as None to auto-detect attached Kaggle datasets.
DATA_PATH = None
FEATURE_DIR = None
NUCLEUS_FEATURE_DIR = None
CHECKPOINT_DIR = None

BUDGETS = [25, 50, 75, 100, 125, 150, 175, 200]
CELL_SOURCE = 'cellvit_embedding'
PCA_DIM = 64
PCA_FIT_SAMPLES = 20000
PROBE_REPEATS = 3
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
OUTPUT_JSON = f'/kaggle/working/nucleus_alignment_{DATASET}_seed{SEED}.json'

In [ ]:
# Resolve and validate every input before training any probe.
import yaml

with open('config/config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
backbone = config.get('models', {}).get('vit', 'facebook/dinov2-base')
safe_vit = backbone.replace('/', '_')
dino_marker = f'{DATASET}_seed{SEED}_{safe_vit}_train.npy'
nucleus_marker = Path(f'{DATASET}_seed{SEED}') / 'manifest.json'
selection_marker = f'{SELECTION_RUN}_selected_budget_{BUDGETS[0]}.pt'

input_root = Path('/kaggle/input')
data_candidates = [
    Path('/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz'),
    Path('/kaggle/input/nckh2026/pathmnist_224.npz'),
]
if input_root.exists():
    data_candidates += list(input_root.rglob('pathmnist_224.npz'))
feature_candidates = [Path('/kaggle/working/features')]
nucleus_candidates = [Path('/kaggle/working/nucleus_features')]
checkpoint_candidates = [Path('/kaggle/working/checkpoints') / DATASET]
if input_root.exists():
    feature_candidates += [p.parent for p in input_root.rglob(dino_marker)]
    nucleus_candidates += [p.parent.parent for p in input_root.rglob(str(nucleus_marker))]
    checkpoint_candidates += [p.parent for p in input_root.rglob(selection_marker)]

def choose_file(explicit, candidates, description):
    if explicit is not None:
        path = Path(explicit)
        assert path.is_file(), f'Missing {description}: {path}'
        return path
    valid = list(dict.fromkeys(str(p) for p in candidates if Path(p).is_file()))
    if not valid:
        raise FileNotFoundError(f'Cannot auto-detect {description}; edit the configuration cell')
    if len(valid) > 1:
        print(f'[warning] multiple {description} candidates; using first:', valid)
    return Path(valid[0])

def choose_dir(explicit, candidates, marker, description):
    if explicit is not None:
        path = Path(explicit)
        assert (path / marker).is_file(), f'Missing {description} marker: {path / marker}'
        return path
    valid = list(dict.fromkeys(str(p) for p in candidates if (Path(p) / marker).is_file()))
    if not valid:
        raise FileNotFoundError(f'Cannot auto-detect {description}; edit the configuration cell')
    if len(valid) > 1:
        print(f'[warning] multiple {description} candidates; using first:', valid)
    return Path(valid[0])

data_path = choose_file(DATA_PATH, data_candidates, 'PathMNIST NPZ')
feature_dir = choose_dir(FEATURE_DIR, feature_candidates, dino_marker, 'DINO feature cache')
nucleus_dir = choose_dir(NUCLEUS_FEATURE_DIR, nucleus_candidates, nucleus_marker, 'nucleus train cache')
checkpoint_dir = choose_dir(CHECKPOINT_DIR, checkpoint_candidates, selection_marker, 'selected-index checkpoints')
assert not OUTPUT_JSON.startswith('/kaggle/input/'), 'OUTPUT_JSON must be writable'
print('data       :', data_path)
print('DINO cache :', feature_dir)
print('cell cache :', nucleus_dir)
print('selections :', checkpoint_dir)
print('device     :', DEVICE)
print('output     :', OUTPUT_JSON)

In [ ]:
# Run the diagnostic. This does not load CellViT or DINO models.
command = [
    sys.executable, 'scripts/evaluate_nucleus_alignment.py',
    '--dataset', DATASET, '--data_path', str(data_path),
    '--seed', str(SEED), '--device', DEVICE,
    '--feature_cache_dir', str(feature_dir),
    '--nucleus_cache_dir', str(nucleus_dir),
    '--checkpoint_dir', str(checkpoint_dir),
    '--cell_source', CELL_SOURCE,
    '--pca_dim', str(PCA_DIM),
    '--pca_fit_samples', str(PCA_FIT_SAMPLES),
    '--probe_repeats', str(PROBE_REPEATS),
    '--output', OUTPUT_JSON,
    '--budgets', *[str(b) for b in BUDGETS],
]
if ALL_RUNS:
    command.append('--all_runs')
else:
    command += ['--selection_run', SELECTION_RUN]
print(' '.join(command))
subprocess.check_call(command)

In [ ]:
# Summarize nAUC and the gain over the equally preprocessed DINO control.
import pandas as pd
from IPython.display import display

result = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
rows = []
for representation, by_run in result['normalized_auc'].items():
    for run_name, nauc in by_run.items():
        dino_control = result['normalized_auc']['dino_normalized'][run_name]
        rows.append({
            'representation': representation,
            'selection_run': run_name,
            'nAUC': nauc,
            'delta_vs_dino_normalized_pp': 100 * (nauc - dino_control),
        })
summary_df = pd.DataFrame(rows).sort_values(['selection_run', 'nAUC'], ascending=[True, False])
display(summary_df.style.format({'nAUC': '{:.4f}', 'delta_vs_dino_normalized_pp': '{:+.3f}'}))

run_name = result['selection_runs'][0]
curve_rows = []
for representation, by_run in result['results'].items():
    for budget in result['budgets']:
        metrics = by_run[run_name][str(budget)]
        curve_rows.append({
            'representation': representation,
            'budget': budget,
            'accuracy_mean': metrics['accuracy_mean'],
            'accuracy_std': metrics['accuracy_std'],
            'macro_f1_mean': metrics['macro_f1_mean'],
        })
curve_df = pd.DataFrame(curve_rows)
display(curve_df.pivot(index='representation', columns='budget', values='accuracy_mean').style.format('{:.4f}'))

In [ ]:
# Heuristic go/no-go decision; confirm any positive result on the official test split.
candidate_names = ['dino_cell_mean', 'dino_cell_moments', 'dino_cell_residual']
for run_name in result['selection_runs']:
    baseline = result['normalized_auc']['dino_normalized'][run_name]
    gains = {name: result['normalized_auc'][name][run_name] - baseline for name in candidate_names}
    best_name = max(gains, key=gains.get)
    best_gain = gains[best_name]
    if best_gain >= 0.005:
        decision = 'GO: extract the nucleus test cache and run official evaluation.'
    elif best_gain <= 0.002:
        decision = 'NO-GO: current cached cell representation is unlikely to justify a larger experiment.'
    else:
        decision = 'INCONCLUSIVE: inspect per-budget gains and repeat with additional acquisition seeds.'
    print(f'{run_name}: best={best_name}, gain={100 * best_gain:+.3f} pp -> {decision}')
print('Reminder: these thresholds are engineering heuristics, not significance tests.')